# Flight compute and competing ecosystems

Companion notebook to [ModalAI VOXL 2 Mini](07_voxl2_mini.ipynb). That one asks
"what can this board do"; this one asks "what else could we build on, and what
would a client buy instead of hiring us".

Three questions, kept separate because they have different answers:

1. **Compute** — what hardware runs the autonomy stack: integrated
   autopilot+computer boards, Jetson plus a separate flight controller, or bare
   Qualcomm modules.
2. **Turnkey platforms** — finished drones and payloads that already do indoor
   GPS-denied capture.
3. **The autonomy question** — which of them actually decides where to fly next,
   as opposed to executing a path an operator drew.

Core take so far: **the compute choice is less interesting than it looks, and
the autonomy gap is real.** Nothing on the market combines autonomous
exploration with photogrammetry-grade imagery — the exploration players are
lidar-first and mining-bred, the imagery players are carried by a human.

Prices are July 2026 and moving fast (see the volatility caveat at the end).

## Why look past the VOXL 2 Mini at all

Not because it is bad — [notebook 07](07_voxl2_mini.ipynb) concludes the
opposite. Because of three practical constraints:

- **Lead time.** The product page states *"Expected to ship within 60 business
  days"* for the Mini, 30 for the full VOXL 2. Roughly three months.
- **Userland age.** Ubuntu 18.04 / kernel 4.19, ROS 2 Foxy, PX4 1.14. Modern
  photogrammetry and SLAM tooling (COLMAP, Open3D, current PyTorch/ONNX) fights
  that, and NPU work goes through Qualcomm's closed SNPE/QNN toolchain.
- **Licence.** Most SDK repos are BSD-3 plus a clause restricting use to ModalAI
  hardware, so nothing lifts onto another board.

Against that, what it uniquely gives is an 11 g complete system with a flight
controller, VIO, mapping and avoidance already integrated — and a real Blue UAS
Framework listing.

## Integrated compute + autopilot boards

The category VOXL invented and still owns.

| Product | Compute | FC | Weight | Price | Buy 1-2? |
|---|---|---|---|---|---|
| [VOXL 2][voxl2] | QRB5165, 15 TOPS | PX4 on DSP | 16 g | $1,299.99 | yes, ~30 days |
| [VOXL 2 Mini][mini] | QRB5165, 15 TOPS | PX4 on DSP | 11 g | $1,249.99 | yes, ~60 days |
| [ARK Pi6X Flow][pi6x] | Pi CM4/CM5, no NPU | ARKV6X | 41 g | $915-1,035 | yes |
| [Airvolute DroneCore Jerboa][jerboa] | Orin NX | STM32H7 FMU | 123 g w/ module | quote only | **sold out** |
| [Airvolute DroneCore 2][dc2] | Orin NX 8/16 GB | Cube or v6X | 206 g | EUR 2,980 | yes (EU) |
| [Sky-Drones AIRLink][airlink] | RK3399-class, no NPU | yes | 89 g | not published | quote only |
| [Auterion Skynode S][skynodes] | quad A53, 2.3 TOPS | FMUv6X | 38 g | not published | **no** |

[voxl2]: https://www.modalai.com/products/voxl-2
[mini]: https://www.modalai.com/products/voxl-2-mini
[pi6x]: https://arkelectron.com/product/ark-pi6x-flow/
[jerboa]: https://docs.airvolute.com/autopilots/dronecore-jerboa
[dc2]: https://docs.airvolute.com/autopilots/dronecore-2
[airlink]: https://docs.sky-drones.com/avionics/airlink/hardware
[skynodes]: https://docs.auterion.com/hardware-integration/skynode-s/datasheet.md

**Auterion is effectively unavailable to us.** No public prices, no store,
Skynode X mission-computer specs unpublished, Skynode N exists only as
[release notes](https://docs.auterion.com/release-notes/auterionos/aos-for-skynode-n.md)
with no product page, and Skynode Enterprise/OEM is
[explicitly discontinued](https://docs.modalai.com/voxl2-mini/). Every path is a
sales conversation aimed at OEM volume. AuterionOS is a closed licensed distro
on top of PX4/MAVLink. Rule it out for research work, not on merit but on
access.

**Airvolute's Jerboa is the closest thing to VOXL-class SWaP in the Jetson
world** — 43 g bare, 123 g with an Orin NX 16 GB, integrated STM32H7 flight
controller, ~8 W. But it is sold out and quote-only, and its
[NDAA declaration explicitly excludes the NVIDIA module](https://airvolute.com/wp-content/uploads/2026/04/2026-04-15-NDAA-Declaration.pdf).
A shop page also lists 221 g shipping weight against the 123 g spec — unresolved.

Note the weight comparison trap: ModalAI's 11 g and 16 g are **complete
systems**. Carrier-board weights from ARK, ConnectTech and Auvidea exclude the
Jetson module (~28 g) and its heatsink and fan (~52 g). The thermal solution
alone outweighs the module.

## The split stack: Pixhawk FC + separate computer

The boring option, and probably the right one for bench research.

| Board | Standard | Weight | Ethernet | Price | NDAA |
|---|---|---|---|---|---|
| [Holybro Pixhawk 6X][h6x] | FMUv6X / PAB | 31.3 g | yes | $166.99 | no (CN) |
| [Holybro Pixhawk 6C mini][h6c] | FMUv6C | ~42 g | no | from $130.99 | no |
| [ARK ARKV6X][arkv6x] | FMUv6X / PAB | **5 g** | via carrier | $400.00 | US-built |
| [Cube Orange+][cube] | CubePilot, not PAB | - | carrier-dep. | $277 | vendor claim |
| [NXP MR-VMU-RT1176][nxp] | FMUv6X-RT | - | yes | $392.70 | 13-week lead |
| [mRo Pixracer Pro][mro] | FMUv5-class | 8.92 g | no | $349.90 | 3-5 wk lead |

[h6x]: https://holybro.com/products/pixhawk-6x
[h6c]: https://holybro.com/products/pixhawk-6c-mini
[arkv6x]: https://arkelectron.com/product/arkv6x/
[cube]: https://irlock.com/products/cube-orange-plus-standard-set
[nxp]: https://www.digikey.com/en/products/detail/nxp-usa-inc/MR-VMU-RT1176/26220861
[mro]: https://store.3dr.com/pixracer-pro/

What the [Pixhawk standard](https://github.com/pixhawk/Pixhawk-Standards) buys:
DS-009 fixes JST-GH connector pinouts so harnesses are cross-vendor; **DS-010
Autopilot Bus (PAB)** makes any compliant FMU drop into any compliant carrier;
DS-012 prescribes the v6X sensor layout so PX4 and ArduPilot targets come free.
Net effect: swap a $167 Holybro FMU for a $400 US-made ARKV6X without rewiring.
**Cube does not give you this** — CubePilot uses its own connector. Ethernet
exists only on v6X-class boards, not 6C.

For a three-week sprint this is the fastest path: a Pixhawk-class FC plus any
Linux box gives Ubuntu 24.04, ROS 2 Jazzy, uXRCE-DDS over Ethernet, and an FC
you can replace same-week. You give up ~150 g and the integrated VIO/mapping
services, which does not matter on a bench or a slightly larger airframe.

**Dead ends worth knowing:** Emlid Navio (Emlid is RTK-only now), Aerotenna
(defunct, hosting suspended), [Aerotenna OcPoC](https://docs.px4.io/main/en/flight_controller/ocpoc_zynq)
(dropped from PX4), Embention Veronte (EUR 23.5k+, wrong league).

## NVIDIA Jetson

The obvious alternative, and it got noticeably weaker in mid-2026.

### Modules

1KU MSRPs after the [22 July 2026 increase][cnxprice]; specs from
[NVIDIA][orin]. The Orin family is supported to January 2032, none EOL.

| Module | TOPS | RAM / BW | Power | Price (old -> new) |
|---|---|---|---|---|
| Orin Nano 4GB | 34 | 4 GB / 51 GB/s | 7-25 W | $199 -> $349 |
| Orin Nano 8GB | 67 | 8 GB / 102 GB/s | 7-25 W | not in the table; post-hike price unverified |
| Orin NX 8GB | 117 | 8 GB / 102.4 GB/s | 10-40 W | $399 -> $649 |
| Orin NX 16GB | 157 | 16 GB / 102.4 GB/s | 10-40 W | $599 -> $999 |
| AGX Orin 32GB | 241 | 32 GB / 204.8 GB/s | 15-60 W | $899 -> $1,799 |
| Thor T5000 | 2070 FP4 TFLOPS | 128 GB | **40-130 W** | ~$2,999, pre-increase |
| Thor T2000 / T3000 | 400 / 865 FP4 TFLOPS | 16 / 32 GB | TBA | **announced only, Q1 2027** |

[cnxprice]: https://www.cnx-software.com/2026/07/22/nvidia-increases-the-price-of-jetson-modules-and-devkits-by-up-to-101/
[orin]: https://www.nvidia.com/en-us/autonomous-machines/embedded-systems/jetson-orin/

Thor T2000/T3000 are the first genuinely drone-relevant Thor parts — and they
ship Q1 2027, with ConnectTech carriers to match. Irrelevant to this project.
The shipping Thors draw 40-130 W, which rules them out for a small indoor
airframe. Channel pricing is inconsistent this week: Arrow and NVIDIA's own buy
page still show pre-increase numbers.

### The software story, which is the real news

**Isaac ROS has moved off Orin.** Isaac ROS 4.5.0 (July 2026) targets ROS 2
Jazzy on JetPack 7.1, and its
[supported-platform table](https://nvidia-isaac-ros.github.io/getting_started/index.html)
lists exactly three things: **Jetson Thor (T4000/T5000), x86_64, and DGX
Spark**. Orin appears nowhere — verified directly, not taken from a summary.
JetPack 7.2 brings Ubuntu 24.04 to Orin, but NVIDIA staff said in June 2026 that
Isaac ROS support is *"coming, in the following months"*.

So the actual Orin stack today is **Isaac ROS 3.2 (Dec 2024) / JetPack 6.2 /
Ubuntu 22.04 / ROS 2 Humble** — 19 months old. The "modern userland" advantage
over VOXL's Ubuntu 18.04 shrinks a lot.

| Component | Honest grade |
|---|---|
| [cuVSLAM][cuvslam] | **Production-usable.** ~116 fps @720p on Orin Nano 8GB, KITTI 0.94% translation error. Wrapper Apache-2.0, **core is a binary blob**. Needs 30 Hz at +/-2 ms jitter — hardware camera/IMU sync is on you, and no ARK/ConnectTech/Auvidea carrier provides it |
| nvblox | **Demo-grade for drones.** Real TSDF/ESDF/mesh, but documented for ground AMRs with Nav2; the costmap is anchored at the robot pose, which breaks for a vehicle at altitude. No aerial tutorial or benchmark |
| Isaac ROS perception | **Genuinely production** TensorRT wrappers (YOLOv8, SegFormer, SAM2, FoundationStereo, AprilTag). Nothing aerial-specific |
| PX4 / ArduPilot integration | **Pure DIY.** PX4's VIO doc still cites the discontinued RealSense T265, PX4-Avoidance is archived, and ARK's own VIO->PX4 demo has not been pushed since Feb 2024 |
| [ARK-OS][arkos] | Active, MIT, but plumbing only: MAVLink routing, RTSP, log upload, DDS bridge. **No VIO, no mapping, no avoidance** |

[cuvslam]: https://github.com/NVIDIA-ISAAC-ROS/isaac_ros_visual_slam
[arkos]: https://github.com/ARK-Electronics/ARK-OS

There is no NVIDIA drone reference architecture. Both headline reasons to pick
Jetson — cheap compute and the Isaac ecosystem — are weaker than they were a
year ago.

### Jetson carriers

The market has consolidated around the Pixhawk Autopilot Bus form factor.
Weights below **exclude** the module (~28 g) and heatsink+fan (~52 g).

| Board | Price | Weight | CSI | Origin / NDAA | Status |
|---|---|---|---|---|---|
| [ARK Just A Jetson][ark-jaj] | $680 | 75 g | 2x | USA, vendor NDAA claim | in stock |
| [ARK Jetson PAB V3][ark-pab3] | $800 | 66 g | 2x | USA | in stock, **but FC breakouts $79.99 not orderable until 1 Sep 2026** |
| [ConnectTech Super Hadron-DM][cti] | $410 | 45 g | 2x 4-lane | Canada, §889/§841 only | **in stock** |
| [Auvidea JNX110][jnx110] | EUR 249-499 | 65 g | 2x | Germany, §889 SKU `70848-NDAA` | in stock |
| [Airvolute Jerboa][jerboa2] | quote | 43 g board | 2x | Slovakia, **excludes the NVIDIA module** | sold out |
| [Holybro Jetson Baseboard][holy] | $395.99 | ~190-203 g | 2x 4-lane | China, none | baseboard only, kits sold out |
| [WeAct N006][weact] | $110.50 | 57.8 g | 2x 4-lane | China, none | AliExpress only |
| [Seeed A603][seeed] | - | 50 g | **1x** | China, none | 1 camera rules out stereo VIO |

[ark-jaj]: https://arkelectron.com/product/ark-just-a-jetson/
[ark-pab3]: https://arkelectron.com/product/ark-jetson-pab-v3/
[cti]: https://www.wdlsystems.com/connect-tech-ngx027
[jnx110]: https://auvidea.eu/product/jnx110-carrier-board-for-nvidia-jetson-orin-nano-nx/
[jerboa2]: https://docs.airvolute.com/autopilots/dronecore-jerboa
[holy]: https://holybro.com/products/pixhawk-jetson-baseboard
[weact]: https://www.cnx-software.com/2026/06/30/weact-n006-a-compact-nvidia-jetson-orin-nx-carrier-board-designed-for-robots-and-uavs/
[seeed]: https://files.seeedstudio.com/products/NVIDIA/A603-Carrier-Board-for-Jetsson-Orin-NX-Nano-Datasheet.pdf

Picks by constraint: **CUDA plus defensible US origin** → ARK Just A Jetson
(built-in ICM-42688P, 3x UART, XT60 direct to 75 V, no custom harness, ~155 g
all-in). **European supply with a PAB slot** → Auvidea JNX110, the only board
here with a written §889 certification form and an integrated STM32F103 running
standard PX4 I/O with 8x PWM. **Weight-constrained research** → ConnectTech
Super Hadron-DM.

### Realistic Jetson brain: BOM and mass

| Item | $ | g |
|---|---|---|
| ARK Just A Jetson carrier | 680 | 75 |
| Orin Nano 8GB module | ~350-500 | 28 |
| Heatsink + fan | ~40 | 52 |
| [ARKV6X FMU](https://arkelectron.com/product/arkv6x/) | 400 | 5 |
| NVMe 240 GB | ~60 | 10 |
| RealSense D435i / D455 (**both out of stock**) | 334-419 | 72-103 |
| CSI stills camera + FFC | ~50-200 | 15-35 |
| Harness, mounts | ~50 | 20-30 |
| **Total** | **~$1,960-2,350** | **~280-340 g** |

Against a VOXL 2 Mini at **$1,249.99 and 11 g** for a complete system with a
flight controller, VIO, stereo depth and mapping already running. The Jetson
brain is roughly 25x the mass and 1.6-1.9x the price, and it hands you no
autonomy stack.

## Bare Qualcomm modules: a dead end

Buying the same QRB5165 from someone else does not get you what VOXL is.

| Product | Price | Status |
|---|---|---|
| [Thundercomm RB5 Core Kit][rb5kit2] | $795.15 | 2020 platform, aging |
| [Lantronix Open-Q 5165RB SOM][openq] | not published | **Last Time Buy — discontinued** |
| [Qualcomm Dragonwing RB3 Gen 2][rb3] (QCS6490, 12 TOPS) | not published | current |
| ADLINK / Excelpoint QRB5165 | - | no current product found |

[rb5kit2]: https://www.arrow.com/en/products/rb5-core-kit/thundercomm.html
[openq]: https://www.lantronix.com/products/open-q-5165rb-som/
[rb3]: https://www.qualcomm.com/developer/hardware/rb3-gen-2-development-kit

**None of these include a flight controller or an IMU.** VOXL's actual
differentiator is PX4 running on the Hexagon DSP with onboard ICM-42688s;
buying a bare SOM means porting that yourself, which is not a three-week task.
QRB5165 is a 2020 part already in end-of-life motion at SOM level. Do not start
new work on it outside VOXL.

## Turnkey indoor platforms — what a client could buy instead

| Product | Sensing | Autonomy, honestly | Price |
|---|---|---|---|
| [Skydio X10][x10] | vision nav | [3D Scan][3dscan] plans its own path inside an operator-defined volume | ~$16-25k, **unverified** |
| [Skydio R10][r10] | vision, zero-light NightSense | operator-flown, avoidance + backtrack. **No mapping product** | not public |
| [Flyability Elios 3][elios] | Ouster OS0-32 + FlyAware SLAM, caged | vendor says [**"Not yet"**][flyauto] autonomous | ~$25k |
| [Emesent Hovermap ST-X][hovermap] | lidar 300 m, Wildcat SLAM | **genuine autonomous exploration** since [Cortex 4.0][cortex] | quote |
| Exyn Nexys | modular lidar + fisheye | "Level 4B" — **their own scale, not a standard** | quote |
| [Leica BLK2FLY][blk2fly] | lidar nav + scan | autonomous inside a drawn boundary; indoor mode retrofitted 2023 | **$44,200** |
| [Cleo Dronut X1][dronut] | 4K + lidar, ducted | assisted flight | $9,800, dated |

[x10]: https://robotomated.com/explore/drone/skydio-x10d
[3dscan]: https://www.skydio.com/software/3d-scan
[r10]: https://www.skydio.com/r10
[elios]: https://robotomated.com/explore/drone/flyability-elios-3
[flyauto]: https://www.flyability.com/software/drone-autonomy
[hovermap]: https://www.emesent.com/autonomy
[cortex]: https://emesent.com/2025/04/29/emesent-enables-fully-autonomous-exploration-and-mapping-of-gps-denied-environments-with-latest-cortex-and-commander-releases/
[blk2fly]: https://shop.leica-geosystems.com/reality-capture/blk2fly/buy
[dronut]: https://uavcoach.com/dronut-x1/

**Vendor health matters here.** Elios 3 launched in 2022 with no successor
announced — a four-year-old platform. Exyn IPO'd in May 2026 raising only
$19.4M, against a $75.9M accumulated deficit and $1.19M quarterly revenue
(-2.3% YoY); treat as a real vendor risk.

**Skydio's autonomy is not available to third parties.** Extend and the Control
& Telemetry ICD expose mission initiation, streaming, telemetry and payload
attachment — not the flight-behaviour stack. There is no autonomy SDK.

### The competitor that is not a drone

| System | Output | Price |
|---|---|---|
| [NavVis VLX 3][vlx] | 2x 32-layer lidar + **4x 20 MP cameras**, +/-5 mm | quote only |
| [Matterport Pro3][pro3] | lidar + 134 MP panoramas — the reference navigable tour | **$5,995** |
| [Leica BLK2GO][blk2go] | handheld SLAM cloud | $60,540 list, $37,550 promo |
| [FARO Orbis][orbis] | SLAM cloud, weak imagery | $19,900 |
| [XGRIDS L2 Pro / PortalCam][xgrids] | **native 3D Gaussian splatting** | EUR 6,295 / 3,799 |

[vlx]: https://knowledge.navvis.com/docs/navvis-vlx-3-specifications
[pro3]: https://www.thefuture3d.com/blog/matterport-pricing-guide-2026/
[blk2go]: https://shop.leica-geosystems.com/reality-capture/blk2go/buy
[orbis]: https://harpersurveying.com/shop/faro-orbis-mobile-laser-scanner/
[xgrids]: https://epotronic.com/eng/manufacturers/xgrids/

A human with a Pro3 or an XGRIDS handheld is the honest competitor for an
occupied building: faster, cheaper, lower-risk than any drone. We should name
that in the report before the client does.

## The autonomy scorecard

The distinction that matters, and that marketing blurs:

- **Autonomous exploration** — the system decides where to go next.
  Emesent Hovermap (bounded-volume), [Verity][verity] and [Corvus One][corvus]
  (warehouse inventory, lights-out, no GPS, no beacons), Exyn (self-declared).
- **Autonomous coverage** — path planning inside an operator-drawn volume.
  Skydio 3D Scan, Leica BLK2FLY. This is what most "autonomous" marketing means.
- **Assisted teleoperation** — Flyability Elios 3 (vendor admits it), Cleo,
  Skydio R10.

[verity]: https://www.verity.net/
[corvus]: https://www.corvus-robotics.com/corvus-one

Verity and Corvus are the proof that indoor autonomy is tractable at scale —
neither produces a 3D deliverable. That is the shape of the gap.

## Where the gap is

1. **Nothing combines autonomous exploration with photogrammetry-grade
   imagery.** Exploration players are lidar-first and mining-bred; imagery
   players are human-carried.
2. **Room-scale, cluttered, furnished interiors are unserved.** Everything
   autonomous is tuned for large open volumes — mines, warehouses, plants.
   Doorways, corridors, furniture, glass and mirrors break both SLAM and
   coverage planning.
3. **Lighting is the unsolved photogrammetry problem indoors.** No vendor claims
   controlled illumination for image quality; onboard lights are for pilot
   visibility and produce harsh view-dependent shading that ruins texture
   consistency.
4. **No autonomous route to the navigable deliverable.** Zillow, CoStar and DJI
   all shipped Gaussian-splat support in 2025, and XGRIDS sells splat-native
   handhelds for EUR 4-6k — but capture is still a human walking with a device.

Items 1 and 2 are precisely where the GLEAM / Next-Best-Path line of work sits,
which is a useful thing to be able to say to the client.

## What this means for our project

- **For three weeks, hardware choice is not the bottleneck** — and the VOXL
  ships the VIO and mapping we would otherwise spend the project rebuilding.
  Its 60-business-day lead time means it is an order-now item regardless.
- **A split stack is faster to iterate on than any integrated board**: a
  Pixhawk-class FC plus any Linux box gives Ubuntu 24.04, ROS 2 Jazzy,
  uXRCE-DDS over Ethernet, and an FC replaceable same-week.
- **Jetson buys CUDA and an open substrate, not autonomy.** With Isaac ROS off
  Orin and no drone reference architecture, you get compute and a build-it-
  yourself stack.
- **The strongest argument against VOXL is roadmap, not capability** — see the
  vendor-direction caveats in [notebook 07](07_voxl2_mini.ipynb).
- **Our differentiator is the exploration policy**, which is hardware-agnostic
  and portable across every platform here through MAVLink and ROS 2.

## Caveats — read before quoting any of this

**NDAA and Blue UAS claims are currently unverifiable.** The DIU Blue UAS
Cleared List [moved to DCMA](https://www.diu.mil/blue-uas-cleared-list) under
the July 2025 Secretary of War memo; `bluelist.dcma.mil` requires sign-in and
could not be enumerated. The
[last public Framework list](https://www.diu.mil/latest/blue-uas-refresh-list-and-framework-platforms-and-capabilities-selected)
(14 entries, March 2025) is the newest public roster. **Every NDAA claim above
is a vendor claim**, and note the fine print differs: ModalAI and ARK cite §848
plus §841; ConnectTech cites §889 and §841 with no manufacturing location;
Auvidea cites §889 only; Airvolute's declaration **excludes the NVIDIA module**.
On that public list, no Jetson carrier appears at all — every compute slot is
held by ModalAI's Qualcomm boards.

**Prices are volatile right now.** NVIDIA raised Jetson prices 33-101% around
22 July 2026 on memory cost pressure, and distributor channels still show
pre-increase figures. Memory pricing also drove LPDDR5X contract prices up ~90%
QoQ in Q1 2026. Anything here older than a few weeks should be re-checked.

**Specifically unverified**, carried forward honestly rather than dropped:
Skydio X10/R10 pricing, NavVis pricing, Dronut current pricing, Exyn's "Level
4B", Orin Nano 8GB post-increase price, Seeed boxed weights, Auvidea
JNX40S/41S/JNX42 dimensions, Airvolute Jerboa price, and whether Auterion
Skynode S is Jetson-based. Two open contradictions left visible rather than
resolved by preference: Holybro 203.2 g vs PX4 docs 190 g for the same build,
and Airvolute Jerboa 123 g spec vs 221 g shop shipping weight.

**Method note.** Compiled from parallel agent research on 2026-07-31, with the
load-bearing claims re-verified directly: VOXL lead time and pricing, the Jetson
price increase, and the Isaac ROS supported-platform table. Where an agent
result conflicted with primary sources, the primary source won.